# Laboratory Exercise 4: A* Algorithm in Python

Members:\
Lamzon, Ellen Faith (2023-60093) \
Malupangue, Carl Cyrill (2023-60041) \
Ringcunada, Ivy (2023-00424)


This program finds the cheapest path from a start point to a goal point
on a small map (graph). Each connection between two points has a cost.\
A* picks which point to visit next using:\
    - f(n) = g(n) + h(n)\
    - g(n) = actual cost from the start to this point\
    - h(n) = estimated (guessed) cost from this point to the goal

## Task 2: Implement A*

In [28]:
## define key variables

initial_state = "S"  # the point where the search starts
goal_state = "G"     # the point we want to reach

# state_space is the "map". For each point, it lists the points it
# connects to and how much it costs to move there.
state_space = {
    "S": {"A": 1, "B": 4},           # from S you can go to A (cost 1) or B (cost 4)
    "A": {"B": 2, "C": 5, "D": 12},  # from A you can go to B, C, or D
    "B": {"C": 2},                   # from B you can only go to C
    "C": {"D": 3, "G": 6},           # from C you can go to D or straight to G
    "D": {"G": 1},                   # from D you can go to G
    "G": {}                          # G is the goal, so it has no outgoing moves
}

# heuristic holds our guess of the remaining cost from each point to G.
# these guesses never go above the real cost, so they are "admissible".
# that is what guarantees A* finds the cheapest path.
heuristic = {
    "S": 7,  # Guess: about 7 more to reach G from S (real cost is 9)
    "A": 6,  # Guess from A (real cost is 8)
    "B": 4,  # Guess from B (real cost is 6)
    "C": 3,  # Guess from C (real cost is 4)
    "D": 1,  # Guess from D (real cost is 1)
    "G": 0   # Already at the goal, so the guess is 0
}

In [29]:
## Node Class
class Node:
    """A Node stores one point in the search plus the costs to reach it."""

    def __init__(self, state, parent=None, g_cost=0, h_cost=0):
        self.state = state                  # the name of the point (e.g., "A")
        self.parent = parent                # the node we came from (used to rebuild the path)
        self.g_cost = g_cost                # actual cost from the start to this node
        self.h_cost = h_cost                # guessed cost from this node to the goal
        self.f_cost = g_cost + h_cost       # total score A* uses to decide what to visit next

    def __repr__(self):
        # Controls how a node looks when printed, e.g. A(g=1, h=6, f=7)
        return f"{self.state}(g={self.g_cost}, h={self.h_cost}, f={self.f_cost})"

In [30]:
## helper function: rebuild the path

def reconstruct_path(node):
    # walk backward from the goal to the start using each node's parent.
    path = []                    # empty list to collect the points
    while node is not None:      # keep going until we pass the start (start has no parent)
        path.append(node.state)  # add the current point to the list
        node = node.parent       # move one step back
    path.reverse()               # the list is goal-to-start, so flip it to start-to-goal
    return path                  # give back the finished path

In [31]:
## the A* Algorithm

def a_star(start, goal, graph, h):
    # search for the cheapest path from start to goal.

    start_node = Node(start, None, 0, h[start])  # make the starting node (g = 0, no parent)

    open_list = [start_node]  # open list = nodes found but not yet explored
    closed_list = set()       # closed list = points already fully explored
    best_g = {start: 0}       # cheapest known cost to reach each point so far

    step = 0  # counts how many nodes we have expanded (for the printout)

    while open_list:  # keep searching as long as there is something left to explore

        # pick the node in the open list with the lowest f_cost.
        # if two are tied, pick the one with the lower h_cost (closer to the goal).
        current = min(open_list, key=lambda n: (n.f_cost, n.h_cost))
        open_list.remove(current)  # take it out of the open list since we are exploring it now

        if current.state in closed_list:  # if this point was already explored...
            continue                      # ...skip it (this is an old, more expensive copy)

        step += 1  # one more node expanded
        print(f"Step {step}: expanding {current}")  # show which node we are working on

        if current.state == goal:  # did we reach the goal?
            return reconstruct_path(current), current.g_cost, step  # return path, total cost, steps

        closed_list.add(current.state)  # mark this point as fully explored

        # look at every neighbor we can move to from the current point
        for neighbor, move_cost in graph[current.state].items():

            if neighbor in closed_list:  # skip neighbors we already explored
                continue

            new_g = current.g_cost + move_cost  # cost to reach the neighbor through the current node

            # only keep this route if it is the first one we found to this neighbor,
            # or if it is cheaper than the route we found before
            if neighbor not in best_g or new_g < best_g[neighbor]:
                best_g[neighbor] = new_g                               # remember the cheaper cost
                child = Node(neighbor, current, new_g, h[neighbor])    # make a new node for the neighbor
                open_list.append(child)                                # add it to the open list to explore later
                print(f"    added {child}")                            # show the new node

    return None, float("inf"), step  # open list ran out and goal was never reached: no path exists

In [32]:
if __name__ == "__main__":  # only runs when this file is run directly
    print("A* Search")                                       # title
    print(f"Start: {initial_state}   Goal: {goal_state}\n")  # show start and goal

    path, total_cost, steps = a_star(initial_state, goal_state, state_space, heuristic)  # run A*

    print("\n----- RESULT -----")
    if path:  # a path was found
        print("Path to goal :", " -> ".join(path))  # print the path like S -> A -> B ...
        print("Total cost   :", total_cost)           # print the total cost of the path
        print("Nodes expanded:", steps)               # print how many nodes A* had to explore
    else:     # no path was found
        print("No path from", initial_state, "to", goal_state)

A* Search
Start: S   Goal: G

Step 1: expanding S(g=0, h=7, f=7)
    added A(g=1, h=6, f=7)
    added B(g=4, h=4, f=8)
Step 2: expanding A(g=1, h=6, f=7)
    added B(g=3, h=4, f=7)
    added C(g=6, h=3, f=9)
    added D(g=13, h=1, f=14)
Step 3: expanding B(g=3, h=4, f=7)
    added C(g=5, h=3, f=8)
Step 4: expanding C(g=5, h=3, f=8)
    added D(g=8, h=1, f=9)
    added G(g=11, h=0, f=11)
Step 5: expanding D(g=8, h=1, f=9)
    added G(g=9, h=0, f=9)
Step 6: expanding G(g=9, h=0, f=9)

----- RESULT -----
Path to goal : S -> A -> B -> C -> D -> G
Total cost   : 9
Nodes expanded: 6


## TASK 3: Apply A* to the 8-Puzzle


# The 8-Puzzle Game

In [33]:
import heapq

def get_valid_moves(state):
    # find where the blank space is right now (index 0 to 8)
    blank_idx = state.index('')

    # quick math to get 2D coordinates (row, col) from the 1D index
    row, col = divmod(blank_idx, 3)

    valid_moves = {}
    # block moves that would push the blank space off the grid or wrap around
    if row > 0: valid_moves['W'] = blank_idx - 3  # Up
    if col > 0: valid_moves['A'] = blank_idx - 1  # Left
    if row < 2: valid_moves['X'] = blank_idx + 3  # Down
    if col < 2: valid_moves['D'] = blank_idx + 1  # Right

    return valid_moves

def apply_move(state, move):
    valid_moves = get_valid_moves(state)
    blank_idx = state.index('')
    target_idx = valid_moves[move]

    # make a copy here so we don't accidentally overwrite the parent node's board
    new_state = state.copy()

    # swap the empty space with the chosen number tile
    new_state[blank_idx], new_state[target_idx] = new_state[target_idx], new_state[blank_idx]

    return new_state

def print_board(state, step_num=None):
    # helper to print the board neatly for the final output
    if step_num is not None:
        print(f"\nStep {step_num}:")
    print("+---+---+---+")
    for r in range(0, 9, 3):
        # convert blanks to actual spaces so the grid doesn't warp
        row_cells = [str(x) if x != '' else ' ' for x in state[r:r+3]]
        print(f"| {row_cells[0]} | {row_cells[1]} | {row_cells[2]} |")
        print("+---+---+---+")

# File Parsing and Heuristic

In [34]:
def read_input_file(filename="input.txt"):
    """Reads the initial and goal configurations from the specified file format."""
    initial_state = []
    goal_state = []
    current_target = None

    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            # switch which list we are appending to based on the header
            if line == "Initial State":
                current_target = initial_state
            elif line == "Goal State":
                current_target = goal_state
            else:
                # convert the comma string to ints, mapping '0' to our standard ''
                row = ['' if x.strip() == '0' else int(x.strip()) for x in line.split(',')]
                current_target.extend(row)

    return initial_state, goal_state

def calculate_manhattan_distance(state, goal_state):
    """Dynamic heuristic function (h_cost) replacing the static dictionary."""
    distance = 0
    for i in range(9):
        val = state[i]
        if val != '':  # we don't count the blank space's distance
            goal_idx = goal_state.index(val)

            # get current x,y and goal x,y to calculate the grid distance
            curr_row, curr_col = divmod(i, 3)
            goal_row, goal_col = divmod(goal_idx, 3)

            # Manhattan distance formula: |x1 - x2| + |y1 - y2|
            distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)

    return distance

# Node Class and Path Reconstruction

In [35]:
class Node:
    """A Node stores one point in the search plus the costs to reach it."""
    def __init__(self, state, parent=None, move=None, g_cost=0, h_cost=0):
        self.state = state                  # The 1D list board
        self.parent = parent                # The node we came from (to trace back the win)
        self.move = move                    # The W/A/X/D move taken to get here
        self.g_cost = g_cost                # Actual cost from start
        self.h_cost = h_cost                # Manhattan guess to goal
        self.f_cost = g_cost + h_cost       # Total score

    def __lt__(self, other):
        # REQUIRED FOR HEAPQ: tells the queue how to break ties.
        # If f_cost is tied, prioritize the one with the lower h_cost (closer to goal).
        if self.f_cost == other.f_cost:
            return self.h_cost < other.h_cost
        return self.f_cost < other.f_cost

def reconstruct_path(node):
    """Walk backward from the goal to the start using each node's parent."""
    path = []
    moves = []

    # keep tracking back until we hit the start node (which has no parent)
    while node is not None:
        path.append(node.state)
        if node.move:
            moves.append(node.move)
        node = node.parent

    # lists are built backwards (goal to start), so flip them
    path.reverse()
    moves.reverse()

    return path, moves

### The A* Search Algorithm

Task 2 used a hardcoded state_space dictionary to map out a tiny graph. However, the 8-puzzle has 181,440 reachable states. Hardcoding all of them would crash this notebook.

For Task 3, we modified the A* implementation to:

Dynamically generate neighbors using our apply_move() function instead of reading a static dictionary.

Calculate the h_cost on the fly using Manhattan distance.

Use Python's heapq (Priority Queue) to instantly find the lowest f_cost node, which prevents the program from hanging.

In [36]:
# --- The A* Search Algorithm ---

def a_star_8_puzzle(start_state, goal_state):
    # setup the first node
    h_start = calculate_manhattan_distance(start_state, goal_state)
    start_node = Node(start_state, None, None, 0, h_start)

    # using a priority queue (heap) instead of a standard list stops the program from freezing
    open_list = []
    heapq.heappush(open_list, start_node)

    closed_list = set()

    # we have to use a tuple for the key because Python dictionaries hate unhashable lists
    best_g = {tuple(start_state): 0}
    nodes_expanded = 0
    max_open_size = 1  # the biggest the open list ever got (we use this to show memory use)

    while open_list:
        # heappop instantly grabs the lowest f_cost node without looping through everything
        current = heapq.heappop(open_list)

        # convert state to tuple so it can go in the set
        current_tuple = tuple(current.state)
        if current_tuple in closed_list:
            continue  # skip if we already explored a cheaper version of this board

        nodes_expanded += 1

        # did we win?
        if current.state == goal_state:
            path, moves = reconstruct_path(current)
            return path, moves, current.g_cost, nodes_expanded, max_open_size

        # mark this board as fully explored
        closed_list.add(current_tuple)

        # dynamically generate the neighbors instead of using a static graph
        valid_moves = get_valid_moves(current.state)
        for move in valid_moves:
            neighbor_state = apply_move(current.state, move)
            neighbor_tuple = tuple(neighbor_state)

            if neighbor_tuple in closed_list:
                continue

            # cost of 1 slide is always 1
            new_g = current.g_cost + 1

            # only add to queue if it's a completely new board or we found a faster way to it
            if neighbor_tuple not in best_g or new_g < best_g[neighbor_tuple]:
                best_g[neighbor_tuple] = new_g
                h = calculate_manhattan_distance(neighbor_state, goal_state)

                # package it into a Node and throw it in the queue
                child = Node(neighbor_state, current, move, new_g, h)
                heapq.heappush(open_list, child)

        # after adding the new boards, check if the open list is bigger than before
        max_open_size = max(max_open_size, len(open_list))

    # if we empty the queue and never hit the goal, the puzzle parity is unsolvable
    return None, None, float("inf"), nodes_expanded, max_open_size

## Board Check

In [37]:
## check 1: is the board valid?

def is_valid_state(state):
    # a valid board must be a list with exactly 9 spots
    if not isinstance(state, list) or len(state) != 9:
        return False

    # a set removes repeated values, so if the set matches exactly,
    # it means every tile 1 to 8 and the blank ('') shows up only once
    return set(state) == {1, 2, 3, 4, 5, 6, 7, 8, ''}


## check 2: can the board be solved?

# half of all 8-puzzle boards can never reach the goal, no matter what moves you do.
# without this check, A* would try all 181,440 boards it can reach before giving up.

def is_solvable(state, goal_state):
    # an "inversion" is a pair of tiles where the bigger number comes before the smaller one.
    # a board can only reach the goal if both have an even number of inversions,
    # or both have an odd number of inversions.

    def count_inversions(board):
        tiles = [t for t in board if t != '']  # remove the blank, we only count the real tiles
        count = 0
        for i in range(len(tiles)):             # look at each tile
            for j in range(i + 1, len(tiles)):  # compare it with every tile after it
                if tiles[i] > tiles[j]:         # bigger number is in front of a smaller one
                    count += 1
        return count

    # % 2 gives 0 if the number is even and 1 if it is odd, so we just compare the two
    return count_inversions(state) % 2 == count_inversions(goal_state) % 2

# Execute

In [38]:
if __name__ == "__main__":
    print("Loading 'input.txt'...")

    # try to read the file. if something is wrong with it, we use empty boards
    # so the validity check below will catch it and show an error message
    try:
        initial_state, goal_state = read_input_file("input.txt")
    except FileNotFoundError:
        # the file is missing
        initial_state, goal_state = [], []
        print("ERROR: input.txt was not found.")
    except (ValueError, AttributeError):
        # ValueError means a tile is not a number, AttributeError means the headers are missing
        initial_state, goal_state = [], []

    # check 1: are both boards real 8-puzzle boards?
    if not is_valid_state(initial_state):
        print("\nERROR: the initial board in input.txt is not valid.")
        print("It must have 9 tiles using the numbers 0 to 8, each one only once.")

    elif not is_valid_state(goal_state):
        print("\nERROR: the goal board in input.txt is not valid.")
        print("It must have 9 tiles using the numbers 0 to 8, each one only once.")

    # check 2: can the initial board reach the goal board?
    elif not is_solvable(initial_state, goal_state):
        print("\nERROR: this puzzle cannot be solved.")
        print("The initial board and the goal board have different inversion parity,")
        print("so no sequence of moves can turn one into the other. Search skipped.")

    # both checks passed, so now we can search
    else:
        print("\nBoard is valid and solvable. Starting A* Search...")

        # a_star_8_puzzle also gives back the biggest open list size, we use it in the performance part
        path, moves, total_cost, expanded, max_open = a_star_8_puzzle(initial_state, goal_state)

        print("\n----- RESULT -----")
        # print the board at each level of the path as requested by Task 3 requirements
        for i, state in enumerate(path):
            if i == 0:
                print_board(state, step_num="Start")
            else:
                print(f"\nMove: {moves[i-1]}")
                print_board(state, step_num=i)

        print(f"\nTotal moves to the goal (Cost): {total_cost}")
        print(f"Nodes expanded: {expanded}")

Loading 'input.txt'...

Board is valid and solvable. Starting A* Search...

----- RESULT -----

Step Start:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 |   | 8 |
+---+---+---+
| 7 | 6 | 5 |
+---+---+---+

Move: X

Step 1:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 6 | 8 |
+---+---+---+
| 7 |   | 5 |
+---+---+---+

Move: D

Step 2:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 6 | 8 |
+---+---+---+
| 7 | 5 |   |
+---+---+---+

Move: W

Step 3:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 6 |   |
+---+---+---+
| 7 | 5 | 8 |
+---+---+---+

Move: A

Step 4:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 |   | 6 |
+---+---+---+
| 7 | 5 | 8 |
+---+---+---+

Move: X

Step 5:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 |   | 8 |
+---+---+---+

Move: D

Step 6:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 | 8 |   |
+---+---+---+

Total moves to the goal (Cost): 6
Nodes expanded: 7


### Performance Analysis

Here we compare A* with BFS (breadth-first search) on four boards, from easy to impossible. For each search we record the path length, how many boards it checked (nodes expanded), the biggest size of the open list (this shows how much memory it used), and how long it took.

In [39]:
## BFS (breadth-first search) for the 8-puzzle

# we copied this from our GUI so we can compare it with A* here in the notebook.
# bfs does not use a heuristic. it just checks the boards level by level,
# so it also finds the shortest path, but it has to check a lot more boards.

from collections import deque  # deque is like a list but it is fast at removing from the front


def bfs_8_puzzle(start_state, goal_state):
    # make the first node from the starting board
    start_node = Node(start_state)

    # if the start is already the goal, there is nothing to search
    if start_state == goal_state:
        return [start_state], [], 0, 0, 1

    # open list = boards waiting to be checked. first in, first out, like a line at a store
    open_list = deque([start_node])

    # visited = boards we already saw, so we don't add the same board twice
    visited = {tuple(start_state)}

    nodes_expanded = 0  # counts how many boards we took out of the line and checked
    max_open_size = 1   # the biggest the open list ever got (we use this to show memory use)

    while open_list:  # keep going while there are still boards in the line
        # take the oldest board from the front of the line
        current = open_list.popleft()
        nodes_expanded += 1

        # try every move the blank can make from this board
        for move in get_valid_moves(current.state):
            neighbor_state = apply_move(current.state, move)
            neighbor_tuple = tuple(neighbor_state)  # turn it into a tuple so it can go in the set

            # skip boards we already saw
            if neighbor_tuple in visited:
                continue
            visited.add(neighbor_tuple)

            # make a node for the new board. it is one move more than the current board
            child = Node(neighbor_state, current, move, current.g_cost + 1)

            # bfs can stop as soon as it sees the goal, because it goes level by level
            # so the first time it sees the goal is already the shortest path
            if neighbor_state == goal_state:
                path, moves = reconstruct_path(child)
                return path, moves, child.g_cost, nodes_expanded, max_open_size

            # new boards go to the back of the line
            open_list.append(child)

        # after adding the new boards, check if the open list is bigger than before
        max_open_size = max(max_open_size, len(open_list))

    # we checked every board we can reach and never found the goal
    return None, None, float("inf"), nodes_expanded, max_open_size

In [41]:
## run the tests and print the table

import time  # we use this to measure how long each search takes

# the first board comes from our input.txt file, the same one we used in task 3
easy_board, goal = read_input_file("input.txt")

# our test boards, from easy to impossible. '' is the blank space.
test_boards = {
    "easy (input.txt)": easy_board,
    "medium":           [8, 1, 3, 4, '', 2, 7, 6, 5],
    "hard":             [8, 6, 7, 2, 5, 4, 3, '', 1],  # needs 31 moves, the most any 8-puzzle can need
    "unsolvable (lab sample)": [1, 2, 3, 8, '', 4, 7, 6, 5],  # the board from the lab slides
}

# the two searches we want to compare
searches = {
    "BFS": bfs_8_puzzle,
    "A*":  a_star_8_puzzle,
}

# print the top row of the table
print(f"{'board':<26}{'search':<8}{'path length':>12}{'nodes expanded':>16}{'max open list':>15}{'time (sec)':>12}")
print("-" * 89)

for board_name, board in test_boards.items():  # go through each test board

    # first check if the board can even be solved
    if not is_solvable(board, goal):
        print(f"{board_name:<26}{'-':<8}{'cannot be solved, so we skip the search':>63}")
        continue  # move on to the next board

    for search_name, search_function in searches.items():  # run bfs, then a*
        start_time = time.perf_counter()  # start the timer
        path, moves, cost, expanded, max_open = search_function(board, goal)  # run the search
        time_taken = time.perf_counter() - start_time  # stop the timer and get how long it took

        # print one row of the table
        print(f"{board_name:<26}{search_name:<8}{cost:>12}{expanded:>16,}{max_open:>15,}{time_taken:>12.4f}")

    print()  # empty line between boards so the table is easier to read

board                     search   path length  nodes expanded  max open list  time (sec)
-----------------------------------------------------------------------------------------
easy (input.txt)          BFS                6              61             53      0.0005
easy (input.txt)          A*                 6               7              9      0.0002

medium                    BFS               14           3,625          2,321      0.0330
medium                    A*                14              39             28      0.0006

hard                      BFS               31         181,349         25,134      0.9549
hard                      A*                31           6,776          3,559      0.0763

unsolvable (lab sample)   -                               cannot be solved, so we skip the search
